# Day 2 실습 코드 — 학습의 원리 · 경사하강 · PyTorch 텐서와 학습 루프

- 강의 자료 [Day 2](https://github.com/Daesung7723/AI-Programming-with-Python/blob/main/Lectures/Day02_%ED%95%99%EC%8A%B5%EC%9B%90%EB%A6%AC_%EA%B2%BD%EC%82%AC%ED%95%98%EA%B0%95_PyTorch.md)의 코드를 강의 순서대로 정리한 노트북입니다.
- 절 제목의 번호는 강의 자료의 절 번호와 같습니다. 개념 설명과 확인 방법은 강의 자료에서 확인합니다.
- 위에서부터 차례로 실행합니다. 뒤의 셀은 앞 셀에서 만든 변수를 사용합니다.
- 응용 실습은 포함하지 않습니다. 강의 자료의 과제를 보고 AI 코딩 도구와 함께 작성합니다.

## 2. 회귀와 손실 함수

### 2.1 직선 하나로 여러 점을 설명하기

**참고 코드**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1.0, 2.0, 3.0, 4.0])
y = 2 * x + 1
print(y)
plt.scatter(x, y)
plt.show()

### 2.2 예측과 정답의 거리 — 손실 함수

**참고 코드**

In [ ]:
W, b = 1.0, 1.0
y_pred = W * x + b
print(y_pred.shape, y_pred)
print(np.mean((y_pred - y) ** 2))

### 2.5 실습 — W를 바꾸며 손실 곡선 그리기

**참고 코드 — 3·4단계**

In [ ]:
W_list = np.linspace(-1, 5, 61)            # ③ -1부터 5까지 61개
losses = []
for W in W_list:                           # ④ W 하나마다 손실 계산
    y_pred = W * x + b
    losses.append(np.mean((y_pred - y) ** 2))   # ⑤ MSE
print(len(losses), losses[0], losses[-1])

plt.plot(W_list, losses)
plt.xlabel("W")
plt.ylabel("MSE")
plt.show()
print(f"손실이 가장 작은 W: {W_list[np.argmin(losses)]:.1f}")   # ⑥

### 2.7 응용 실습 — 손실이 0이지만 부합하지 않는 직선

> 응용 실습 2.7에서 원인을 찾는 대상 코드입니다.

In [ ]:
loss = np.mean(y_pred - y) ** 2

## 3. 경사하강

### 3.1 손실을 줄이는 방향

**참고 코드**

In [ ]:
w = 0.0
for step in range(3):
    w = w - 0.1 * 2 * (w - 3)
    print(f"{step + 1}번째 갱신 후 w = {w:.3f}")

### 3.5 실습 — 경사하강을 직접 실행하고 학습률 비교하기

**참고 코드 — 2·3단계**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def loss(w):                          # ① 손실 함수
    return (w - 3) ** 2

def grad(w):                          # ② 수기로 구한 도함수
    return 2 * (w - 3)

for lr in [0.01, 0.1, 1.1]:           # ③ 학습률 세 가지
    w = 0.0
    history = []
    for step in range(20):
        w = w - lr * grad(w)          # ④ 갱신 규칙 — 3.1의 식
        history.append(loss(w))
    plt.plot(history, label=f"lr={lr}")
    print(f"lr={lr}: 최종 w = {w:.3f}")

plt.yscale("log")                     # ⑤ 손실 차이가 커서 로그 눈금
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## 4. 텐서 기초

### 4.2 텐서 생성과 속성

In [ ]:
import torch
import numpy as np

x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])   # ① 리스트에서
z = torch.zeros(2, 3)                                  # ② 0으로 채움
r = torch.randn(2, 3)                                  # ③ 표준정규분포 난수
o = torch.ones_like(x)                                 # ④ x와 같은 형상·자료형

a = np.array([7, 8, 9])
t = torch.from_numpy(a)                                # ⑤ NumPy에서 — 메모리 공유

print(x.shape, x.dtype, x.device)                      # ⑥ 속성 세 가지
print(x.numel())                                       # ⑦ 원소 개수

**모듈을 가져오는 두 가지 형태**

In [ ]:
import torch.nn as nn        # 형태 ① import 모듈 as 별칭
from torch import nn         # 형태 ② from 모듈 import 이름

### 4.3 GPU 이동과 연산 시간 비교

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"현재 장치: {device}")

In [ ]:
import time
size = 5000
a = torch.randn(size, size)
b = torch.randn(size, size)

start = time.time()
torch.matmul(a, b)                          # ① CPU에서 계산
print(f"CPU: {time.time() - start:.3f}초")

if device == "cuda":
    a_gpu = a.to(device)                    # ② GPU로 복사
    b_gpu = b.to(device)
    torch.cuda.synchronize()                # ③ 이전 작업이 끝날 때까지 대기
    start = time.time()
    torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()
    print(f"GPU: {time.time() - start:.3f}초")

> GPU가 배정된 경우에만 실행합니다. CPU 장치이면 `NameError`가 출력되며, 이 셀을 건너뜁니다(강의 자료 4.3).

In [ ]:
back = a_gpu.cpu()          # ④ CPU로 복사
arr = back.numpy()          # ⑤ NumPy 배열로 변환
print(type(arr), arr.shape)

### 4.4 형상 조작

In [ ]:
t = torch.arange(12, dtype=torch.float32).view(3, 4)   # ① 0~11을 3행 4열로
print(t.view(2, 6).shape)                  # ② (2, 6)
print(t.view(-1, 2).shape)                 # ③ -1 = 자동 계산 → (6, 2)
print(t.unsqueeze(0).shape)                # ④ (1, 3, 4)
print(t.unsqueeze(0).squeeze(0).shape)     # ⑤ (3, 4)

**2단계 — 이미지의 축 순서 바꾸기**

In [ ]:
img = torch.rand(10, 12, 3)                # ⑥ (H, W, C) 순서의 가상 이미지
chw = img.permute(2, 0, 1)                 # ⑦ (C, H, W)로 축 순서 교체
batch = chw.unsqueeze(0)                   # ⑧ (1, C, H, W) — 배치 축 추가
print(chw.shape, batch.shape)

small = torch.arange(36).view(3, 4, 3)     # ⑨ 값으로 위치를 추적하는 작은 (H, W, C)
chw_s = small.permute(2, 0, 1)
print(small[0, 0])
print(chw_s[:, 0, 0])
print(chw_s[0])

**3단계 — 이어 붙이기와 축 방향 계산**

In [ ]:
t1 = torch.ones(2, 3)
t2 = torch.zeros(2, 3)
print(torch.cat([t1, t2], dim=0).shape)     # (4, 3) — 0번 축으로 이어 붙임
print(torch.cat([t1, t2], dim=1).shape)     # (2, 6) — 1번 축으로 이어 붙임
print(torch.stack([t1, t2], dim=0).shape)   # (2, 2, 3) — 새 축을 만들어 쌓음

print(t.sum())                              # 전체 합
print(t.sum(dim=0))                         # 0번 축을 따라 — 열마다 합
print(t.mean(dim=1))                        # 1번 축을 따라 — 행마다 평균
print(t.argmax(dim=1))                      # 행마다 가장 큰 값의 위치

## 5. autograd와 역전파

### 5.2 `requires_grad`와 `backward()` — 수기 계산과 대조

In [ ]:
import torch

x = torch.tensor(1.0)                       # 입력 — 학습 대상 아님
y = torch.tensor(2.0)                       # 정답 — y = 2x
w = torch.tensor(1.0, requires_grad=True)   # ① 학습할 가중치 — 기울기 추적

y_pred = w * x                              # ② 순전파
loss = (y_pred - y) ** 2                    # ③ 손실

manual = 2 * (w * x - y) * x                # ④ 수기로 구한 도함수
print(f"수기 계산: {manual.item():.2f}")
loss.backward()                             # ⑤ 역전파
print(f"autograd: {w.grad.item():.2f}")     # ⑥ 계산된 기울기

w_new = w.item() - 0.1 * w.grad.item()      # ⑦ 경사하강 한 단계
print(f"갱신 후 w: {w_new:.2f}")

### 5.3 기울기의 누적과 초기화

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
for i in range(2):
    loss = w * w
    loss.backward()
    print(f"{i + 1}번째 backward 후 w.grad: {w.grad}")

> 따로 실행하는 코드가 아닙니다. 위 셀의 반복문 안 `loss.backward()` 바로 앞에 추가하는 두 줄입니다(강의 자료 5.3).

```python
    if w.grad is not None:
        w.grad.zero_()
```

### 5.4 잎 노드·스칼라 손실·`.item()`

In [ ]:
a = torch.tensor(1.0, requires_grad=True)   # 잎 노드 — 사용자가 직접 생성
h = a * 2                                   # 중간 노드 — 연산 결과
z = h * 3
z.backward()
print(a.grad)       # tensor(6.)
print(h.grad)       # None

In [ ]:
loss_history = []
loss_history.append(loss.item())    # 6절·8절에서 사용하는 기록 방식

### 5.6 확인 활동 — 출력 예측

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
loss = (w - 1) ** 2
loss.backward()
print(w.grad)       # (가)
loss = (w - 1) ** 2
loss.backward()
print(w.grad)       # (나)

### 5.7 실습 — 두 파라미터의 기울기 수기 계산과 대조

**참고 코드**

In [ ]:
x = torch.tensor(2.0)
y = torch.tensor(5.0)
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

loss = (w * x + b - y) ** 2
loss.backward()
print(w.grad, b.grad)                          # ② 수기 계산 결과와 대조

w_new = w.item() - 0.05 * w.grad.item()        # ③ 한 단계 갱신
b_new = b.item() - 0.05 * b.grad.item()
print(f"갱신 후 w = {w_new:.2f}, b = {b_new:.2f}")
new_loss = (w_new * 2 + b_new - 5) ** 2
print(f"손실 {loss.item():.2f} → {new_loss:.2f}")    # 갱신 전후 손실

## 6. 학습 루프 5단

### 6.1 직접 작성하는 학습 루프

**1단계 — 데이터와 시작값**

In [ ]:
import torch
torch.manual_seed(0)

x_train = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y_train = torch.tensor([[3.0], [5.0], [7.0], [9.0]])
W = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
print(x_train.shape, W.shape)
print(W.item(), b.item())
y_pred = x_train * W + b
print(y_pred.shape)

**2단계 — 갱신 한 번**

In [ ]:
loss = torch.mean((y_pred - y_train) ** 2)    # 손실(MSE)
loss.backward()                               # 역전파
print(loss.item(), W.grad.item(), b.grad.item())
with torch.no_grad():                         # 갱신 — 그래프에 기록하지 않음
    W -= 0.01 * W.grad
    b -= 0.01 * b.grad
print(W.item(), b.item())

In [ ]:
import torch
torch.manual_seed(0)                              # 난수 고정 — 실행마다 같은 결과

x_train = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y_train = torch.tensor([[3.0], [5.0], [7.0], [9.0]])   # y = 2x + 1

W = torch.randn(1, requires_grad=True)            # ① 무작위 시작값
b = torch.randn(1, requires_grad=True)
lr = 0.01

for epoch in range(2000):
    y_pred = x_train * W + b                      # ② 순전파
    loss = torch.mean((y_pred - y_train) ** 2)    # ③ 손실(MSE)

    if W.grad is not None:                        # ④ 기울기 초기화
        W.grad.zero_()
        b.grad.zero_()
    loss.backward()                               # ⑤ 역전파

    with torch.no_grad():                         # ⑥ 갱신 — 그래프에 기록하지 않음
        W -= lr * W.grad
        b -= lr * b.grad

    if (epoch + 1) % 200 == 0:                    # ⑦ 200회마다 출력
        print(f"epoch {epoch + 1:4d} | 손실 {loss.item():.4f} | W {W.item():.3f} | b {b.item():.3f}")

### 6.2 옵티마이저 3단 호출 — `zero_grad → backward → step`

In [ ]:
torch.manual_seed(0)
W = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
optimizer = torch.optim.SGD([W, b], lr=0.01)      # 갱신할 파라미터 목록 · 학습률

for epoch in range(2000):
    y_pred = x_train * W + b                      # ① 순전파
    loss = torch.mean((y_pred - y_train) ** 2)    # ② 손실
    optimizer.zero_grad()                         # ③ 기울기 초기화
    loss.backward()                               # ④ 역전파
    optimizer.step()                              # ⑤ 갱신

print(f"W {W.item():.3f} | b {b.item():.3f}")

## 7. nn.Module·활성화·옵티마이저

### 7.1 `nn.Module`의 구조

In [ ]:
import torch.nn as nn

class LinearModel(nn.Module):                 # ① nn.Module을 상속
    def __init__(self):
        super().__init__()                    # ② 부모 클래스의 준비 작업 실행
        self.linear = nn.Linear(1, 1)         # ③ 입력 1 → 출력 1

    def forward(self, x):                     # ④ 데이터가 지나는 순서
        return self.linear(x)

model = LinearModel()                         # ⑤ 객체 생성
print(model)
print(model.linear.weight, model.linear.bias)

### 7.2 비선형이 필요한 이유 — 활성화 없이 쌓기

In [ ]:
import matplotlib.pyplot as plt
torch.manual_seed(0)
x = torch.linspace(-3, 3, 100).unsqueeze(1)                   # (100, 1)
y = 0.5 * x ** 2 - 2 * x + 1 + 0.3 * torch.randn(100, 1)      # 곡선 + 잡음

def train(model, epochs=1000):                                 # 학습 함수
    opt = torch.optim.Adam(model.parameters(), lr=0.01)       # Adam = 7.4
    for _ in range(epochs):
        loss = torch.mean((model(x) - y) ** 2)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item()

no_act = nn.Sequential(nn.Linear(1, 16), nn.Linear(16, 1))
with_act = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
print(f"활성화 없음: {train(no_act):.3f} · ReLU 포함: {train(with_act):.3f}")

plt.scatter(x, y, s=8)
plt.plot(x, no_act(x).detach(), label="no activation")
plt.plot(x, with_act(x).detach(), label="ReLU")
plt.legend()
plt.show()

### 7.3 활성화 함수 4종과 기울기 소실

**실습 — 활성화 함수와 도함수 그리기**

In [ ]:
funcs = {"sigmoid": torch.sigmoid, "tanh": torch.tanh,
         "ReLU": torch.relu, "LeakyReLU": nn.functional.leaky_relu}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name, f in funcs.items():
    z = torch.linspace(-6, 6, 200, requires_grad=True)
    out = f(z)
    out.sum().backward()                     # 합의 기울기 = 원소별 도함수
    axes[0].plot(z.detach(), out.detach(), label=name)
    axes[1].plot(z.detach(), z.grad, label=name)
axes[0].set_title("function")
axes[1].set_title("derivative")
axes[0].legend()
plt.show()

### 7.4 옵티마이저의 세대

**참고 코드**

In [ ]:
def make_model():
    torch.manual_seed(0)                           # 두 모델의 시작값을 같게
    return nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))

for name in ["SGD", "Adam"]:
    model = make_model()
    if name == "SGD":
        opt = torch.optim.SGD(model.parameters(), lr=0.01)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=0.01)
    history = []
    for _ in range(500):
        loss = torch.mean((model(x) - y) ** 2)
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.item())
    plt.plot(history, label=name)
    print(f"{name}: 10스텝 {history[9]:.3f} · 50스텝 {history[49]:.3f} · 500스텝 {history[-1]:.3f}")

plt.yscale("log")
plt.legend()
plt.show()

### 7.5 가중치 초기화

In [ ]:
layer = nn.Linear(16, 1)
print(layer.weight.min().item(), layer.weight.max().item())   # 가중치 16개의 최솟값·최댓값
print(1 / 16 ** 0.5)                                            # 범위의 끝 √k

### 7.6 `nn.Module`로 곡선 피팅 모델 작성

In [ ]:
class CurveNet(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.layer1 = nn.Linear(1, hidden)        # ① 입력 1 → 은닉 hidden
        self.act = nn.ReLU()                      # ② 활성화
        self.layer2 = nn.Linear(hidden, 1)        # ③ 은닉 hidden → 출력 1

    def forward(self, x):
        x = self.layer1(x)
        x = self.act(x)
        return self.layer2(x)

model = CurveNet()
loss_fn = nn.MSELoss()                                          # ④ 손실 함수 객체
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)      # ⑤ 옵티마이저
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {n_params}")

In [ ]:
a1 = model.layer1(x)
a2 = model.act(a1)
a3 = model.layer2(a2)
print(a1.shape, a2.shape, a3.shape)
print((a1 < 0).float().mean().item(), (a2 == 0).float().mean().item())

## 8. Day 산출물 — 곡선 피팅 신경망

### 8.2 필수 — 학습과 결과 확인

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
x = torch.linspace(-3, 3, 100).unsqueeze(1)                  # ① (100, 1)
y = 0.5 * x ** 2 - 2 * x + 1 + 0.3 * torch.randn(100, 1)     # ② 곡선 + 잡음

model = CurveNet()                                            # ③ 7.6의 클래스
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

loss_history = []
for epoch in range(1000):
    y_pred = model(x)                       # 순전파
    loss = loss_fn(y_pred, y)               # 손실
    optimizer.zero_grad()                   # 기울기 초기화
    loss.backward()                         # 역전파
    optimizer.step()                        # 갱신
    loss_history.append(loss.item())        # ④ 기록

print(f"최종 손실: {loss_history[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(loss_history)
axes[0].set_yscale("log")
axes[0].set_title("loss")
axes[1].scatter(x, y, s=8, label="data")
axes[1].plot(x, model(x).detach(), color="red", label="model")
axes[1].legend()
plt.show()

## 10. 보충

### 10.2 기울기 소실의 수치

In [ ]:
for name, f in [("sigmoid", torch.sigmoid), ("ReLU", torch.relu)]:
    for n in [1, 3, 5, 10]:
        x = torch.tensor(1.0, requires_grad=True)
        h = x
        for _ in range(n):
            h = f(h)                 # 활성화를 n번 겹쳐 적용
        h.backward()
        print(f"{name} {n:2d}회: 기울기 {x.grad.item():.2e}")

### 10.5 다중 선형 회귀 — 특징이 늘면 행렬 곱

In [ ]:
torch.manual_seed(0)
x_train = torch.tensor([[1.0, 1.0], [2.0, 3.0], [3.0, 2.0], [4.0, 5.0]])   # (4, 2)
y_train = torch.tensor([[5.5], [13.5], [12.5], [23.5]])    # y = 2x₁ + 3x₂ + 0.5

W = torch.randn((2, 1), requires_grad=True)     # 특징 수만큼 — (2, 1)
b = torch.randn(1, requires_grad=True)
optimizer = torch.optim.SGD([W, b], lr=0.01)

for epoch in range(2000):
    y_pred = x_train @ W + b                    # (4, 2) @ (2, 1) → (4, 1)
    loss = torch.mean((y_pred - y_train) ** 2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(W.squeeze().tolist(), b.item())           # [2, 3] · 0.5 부근

### 10.6 학습 과정 시각화 — 슬라이더

> 슬라이더는 노트북 화면(Colab 등)에서 동작합니다.

In [ ]:
import torch
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def train_and_plot(epochs, lr):
    x, y = torch.tensor(1.0), torch.tensor(2.0)       # 정답 w = 2
    w = torch.tensor(1.0, requires_grad=True)
    w_hist, loss_hist = [], []
    for _ in range(epochs):
        loss = (w * x - y) ** 2
        w_hist.append(w.item())
        loss_hist.append(loss.item())
        if w.grad is not None:
            w.grad.zero_()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(loss_hist)
    axes[0].set_title("loss")
    axes[1].plot(w_hist)
    axes[1].axhline(2.0, color="red", linestyle="--")
    axes[1].set_title("w")
    plt.show()

interact(train_and_plot,
         epochs=IntSlider(min=10, max=500, step=10, value=100),
         lr=FloatSlider(min=0.001, max=1.2, step=0.001, value=0.01, readout_format=".3f"))